# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
Looking at the data, the traffic metrics (impressions and sessions) are heavily skewed. Most pages get very little traffic, while a few get a massive amount (heavy right tail). Content age is more spread out but still shows variations.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Define our target label
df['is_declining'] = df['trend_direction'] == 'down'

print("Checking percentiles to see the heavy tails in traffic:")
print(df[['impressions_90d', 'sessions_90d', 'content_age_days']].quantile([0.5, 0.75, 0.95]))

Checking percentiles to see the heavy tails in traffic:
      impressions_90d  sessions_90d  content_age_days
0.50           731.00           7.0             236.0
0.75          3615.25          27.0             333.0
0.95         22996.50         166.0             487.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal 1 (Age): Older pages decay more often. Verdict: CONFIRMED (The decay rate goes up as pages get older).

Signal 2 (Impressions): Low impression pages decay faster. Verdict: MIXED (Decay happens across the board; even high-traffic pages drop, likely due to regression to the mean).

Signal 3 (Sessions): Pages with low sessions are more likely to decline. Verdict: MIXED (Similar to impressions, the pattern isn't a straight line).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Signal 1: Age vs Decay")
df['age_bin'] = pd.qcut(df['content_age_days'], 3, labels=['New', 'Medium', 'Old'])
print(df.groupby('age_bin')['is_declining'].mean().round(3))

print("\nSignal 2: Impressions vs Decay")
# Adding 1 to avoid duplicate bin edges for zeros
df['imp_bin'] = pd.qcut(df['impressions_90d'] + 1, 3, labels=['Low', 'Medium', 'High'])
print(df.groupby('imp_bin')['is_declining'].mean().round(3))


Signal 1: Age vs Decay
age_bin
New       0.625
Medium    0.561
Old       0.437
Name: is_declining, dtype: float64

Signal 2: Impressions vs Decay
imp_bin
Low       0.434
Medium    0.611
High      0.581
Name: is_declining, dtype: float64


/tmp/ipykernel_3357/2664275516.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('age_bin')['is_declining'].mean().round(3))
/tmp/ipykernel_3357/2664275516.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('imp_bin')['is_declining'].mean().round(3))


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Testing a common SEO rule: "Pages older than 365 days are decaying and need a refresh."
Looking at the data, pages over a year old do have a higher decline rate compared to newer ones. So the basic assumption directionally holds, but it's not a 100% guarantee.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['is_older_than_year'] = df['content_age_days'] > 365

print("Decay rate for pages > 365 days old vs newer pages:")
print(df.groupby('is_older_than_year')['is_declining'].mean().round(3))
print("Verdict: The assumption holds directionally, but isn't an absolute rule.")

Decay rate for pages > 365 days old vs newer pages:
is_older_than_year
False    0.573
True     0.426
Name: is_declining, dtype: float64
Verdict: The assumption holds directionally, but isn't an absolute rule.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In practice, the content team shouldn't just use a single rule like "update everything over a year old." While age is a good signal, traffic drops happen everywhere. A combined ML score will help them prioritize the most urgent pages much better than simple if-statements.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Takeaway: Single metrics aren't enough.")
print("We need a combined ML model to rank the review queue effectively.")


Takeaway: Single metrics aren't enough.
We need a combined ML model to rank the review queue effectively.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.